# Mini Project 01 — Titanic Classification
## Section A — Problem Definition & Data Preparation

**Deliverable:** `A_minimal_prep_and_split.ipynb` &nbsp;|&nbsp; **Output:** `X_train, X_test, y_train, y_test`

### Target selection / prediction objective
> **Predict whether a passenger survived the Titanic sinking (`Survived` = 1) or not (`Survived` = 0) from their class, sex, age, family size, fare and port of embarkation.**

`Survived` is a binary, meaningful classification target, and the project question — *"what sorts of people were more likely to survive?"* — maps directly onto it.

### Pipeline (minimal, leakage-safe)
1. Load the Kaggle Titanic data.
2. Drop obvious identifiers and free-text columns (`PassengerId`, `Name`, `Ticket`, `Cabin`).
3. **Stratified 80/20 split first** (seed 42) — so that every statistic used for imputation / scaling is learned from the training set only (no leakage).
4. Simple imputation: median (`Age`, `Fare`), mode (`Embarked`).
5. Encode categoricals: `Sex` → binary, `Pclass` & `Embarked` → one-hot (reference level dropped).
6. Light scaling: `log1p(Fare)`, then standardise the numeric columns with train mean/std (needed for gradient-descent logistic regression in Section B).
7. Save the processed arrays so Sections B and C use the **identical** split.

In [1]:
import os, sys, platform
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split   # Section A may use sklearn (restriction applies to Section B only)

SEED = 42                      # global seed used in ALL notebooks
np.random.seed(SEED)

print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

Python 3.12.3 | numpy 2.4.4 | pandas 3.0.2
scikit-learn 1.8.0


## 1. Load the data

In [2]:
DATA_PATH = "data/titanic.csv"
URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"  # mirror of the Kaggle train.csv

if not os.path.exists(DATA_PATH):               # fallback if the CSV was not shipped next to the notebook
    os.makedirs("data", exist_ok=True)
    pd.read_csv(URL).to_csv(DATA_PATH, index=False)

df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
df.head()

shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0].to_string())
print(f"\nOverall survival rate: {df['Survived'].mean():.3f}")
print("\nSurvival rate by Sex:");    print(df.groupby('Sex')['Survived'].mean().round(3).to_string())
print("\nSurvival rate by Pclass:"); print(df.groupby('Pclass')['Survived'].mean().round(3).to_string())

Missing values per column:
Age         177
Cabin       687
Embarked      2

Overall survival rate: 0.384

Survival rate by Sex:
Sex
female    0.742
male      0.189

Survival rate by Pclass:
Pclass
1    0.630
2    0.473
3    0.242


## 2. Drop identifiers and free-text columns
* `PassengerId` – row identifier, no predictive meaning.
* `Name` – free text (titles such as *Mr/Mrs* mostly duplicate `Sex`/`Age`; we deliberately keep preprocessing minimal).
* `Ticket` – free-text/alphanumeric identifier.
* `Cabin` – free text and ~77 % missing.

In [4]:
DROP_COLS = ["PassengerId", "Name", "Ticket", "Cabin"]
TARGET = "Survived"

y = df[TARGET].astype(int)
X_raw = df.drop(columns=DROP_COLS + [TARGET])
print("Remaining feature columns:", list(X_raw.columns))

Remaining feature columns: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']


## 3. Stratified 80/20 train/test split (done *before* imputation/scaling)
`stratify=y` guarantees the survival rate is (almost) identical in both parts. The same split is re-used by Sections B and C.

In [5]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.20, stratify=y, random_state=SEED
)
print(f"train: {X_train_raw.shape}   test: {X_test_raw.shape}")
print(f"survival rate  full={y.mean():.4f}  train={y_train.mean():.4f}  test={y_test.mean():.4f}")

train: (712, 7)   test: (179, 7)
survival rate  full=0.3838  train=0.3834  test=0.3855


## 4. Simple imputation (statistics learned on the **training** set only)
| Column | Strategy | Why |
|---|---|---|
| `Age` | median | robust to the right-skewed age distribution |
| `Fare` | median | robust to extreme fares |
| `Embarked` | mode | categorical, only 2 missing |

In [6]:
impute_values = {
    "Age":      float(X_train_raw["Age"].median()),
    "Fare":     float(X_train_raw["Fare"].median()),
    "Embarked": str(X_train_raw["Embarked"].mode()[0]),
}
print("Imputation values (from train):", impute_values)

X_train_imp = X_train_raw.fillna(impute_values)
X_test_imp  = X_test_raw.fillna(impute_values)      # test set uses TRAIN statistics
print("Remaining NaNs  train:", int(X_train_imp.isna().sum().sum()), " test:", int(X_test_imp.isna().sum().sum()))

Imputation values (from train): {'Age': 28.5, 'Fare': 14.4542, 'Embarked': 'S'}
Remaining NaNs  train: 0  test: 0


## 5. Encode categorical variables
* `Sex` → single binary column `Sex_male` (1 = male).
* `Pclass` (1/2/3, a *class*, not a quantity) → one-hot, reference = 1st class.
* `Embarked` (C/Q/S) → one-hot, reference = Cherbourg (C).

Fixed category lists are used so train and test always end up with **exactly the same columns**.

In [7]:
def encode(X):
    out = pd.DataFrame(index=X.index)
    out["Age"]      = X["Age"].astype(float)
    out["SibSp"]    = X["SibSp"].astype(float)
    out["Parch"]    = X["Parch"].astype(float)
    out["Fare"]     = np.log1p(X["Fare"].astype(float))        # log1p tames the heavy right tail of Fare
    out["Sex_male"] = (X["Sex"] == "male").astype(float)
    for c in (2, 3):                                             # reference level: 1st class
        out[f"Pclass_{c}"] = (X["Pclass"] == c).astype(float)
    for c in ("Q", "S"):                                         # reference level: Cherbourg
        out[f"Embarked_{c}"] = (X["Embarked"] == c).astype(float)
    return out

X_train_enc, X_test_enc = encode(X_train_imp), encode(X_test_imp)
assert list(X_train_enc.columns) == list(X_test_enc.columns)
X_train_enc.head()

,Age,SibSp,Parch,Fare,Sex_male,Pclass_2,Pclass_3,Embarked_Q,Embarked_S
692,28.5,0.0,0.0,4.051712,1.0,0.0,1.0,0.0,1.0
481,28.5,0.0,0.0,0.000000,1.0,1.0,0.0,0.0,1.0
527,28.5,0.0,0.0,5.406181,1.0,0.0,0.0,0.0,1.0
855,18.0,0.0,1.0,2.336987,0.0,0.0,1.0,0.0,1.0
801,31.0,1.0,1.0,3.305054,0.0,1.0,0.0,0.0,1.0


## 6. Scale numeric columns (train mean / std)
Binary dummy columns are left untouched. Scaling makes gradient descent in Section B well-conditioned and puts the L1/L2 penalties on a comparable footing for every feature.

In [8]:
NUM_COLS = ["Age", "SibSp", "Parch", "Fare"]
mu, sd = X_train_enc[NUM_COLS].mean(), X_train_enc[NUM_COLS].std(ddof=0)

X_train, X_test = X_train_enc.copy(), X_test_enc.copy()
X_train[NUM_COLS] = (X_train[NUM_COLS] - mu) / sd
X_test[NUM_COLS]  = (X_test[NUM_COLS]  - mu) / sd      # train statistics only

print("train means (numeric):", X_train[NUM_COLS].mean().round(3).to_dict())
print("train stds  (numeric):", X_train[NUM_COLS].std(ddof=0).round(3).to_dict())
X_train.describe().T.round(3)

train means (numeric): {'Age': 0.0, 'SibSp': -0.0, 'Parch': -0.0, 'Fare': 0.0}
train stds  (numeric): {'Age': 1.0, 'SibSp': 1.0, 'Parch': 1.0, 'Fare': 1.0}


,count,mean,std,min,25%,50%,75%,max
Age,712.0,0.000,1.001,-2.238,-0.581,-0.081,0.495,3.875
SibSp,712.0,-0.000,1.001,-0.465,-0.465,-0.465,0.478,7.082
Parch,712.0,-0.000,1.001,-0.466,-0.466,-0.466,-0.466,6.698
Fare,712.0,0.000,1.001,-3.014,-0.782,-0.218,0.526,3.361
Sex_male,712.0,0.645,0.479,0.000,0.000,1.000,1.000,1.000
Pclass_2,712.0,0.211,0.408,0.000,0.000,0.000,0.000,1.000
Pclass_3,712.0,0.549,0.498,0.000,0.000,1.000,1.000,1.000
Embarked_Q,712.0,0.077,0.267,0.000,0.000,0.000,0.000,1.000
Embarked_S,712.0,0.728,0.446,0.000,0.000,1.000,1.000,1.000


## 7. Sanity checks

In [9]:
assert not X_train.isna().any().any() and not X_test.isna().any().any()
assert X_train.shape[1] == X_test.shape[1]
assert len(X_train) == len(y_train) and len(X_test) == len(y_test)
assert set(X_train.index).isdisjoint(X_test.index)           # no row appears in both sets
assert abs(y_train.mean() - y_test.mean()) < 0.01            # stratification worked

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)
print("Features:", list(X_train.columns))
print(f"Train class balance: {y_train.mean():.4f} survived | Test: {y_test.mean():.4f} survived")

X_train: (712, 9) | X_test: (179, 9)
y_train: (712,) | y_test: (179,)
Features: ['Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Pclass_2', 'Pclass_3', 'Embarked_Q', 'Embarked_S']
Train class balance: 0.3834 survived | Test: 0.3855 survived


## 8. Save the processed datasets
Saved both as human-readable CSVs and as a single `.npz` (what Sections B and C load), so every later section uses the *identical* split.

In [10]:
OUT = "data/processed"
os.makedirs(OUT, exist_ok=True)

X_train.to_csv(f"{OUT}/X_train.csv", index=False); X_test.to_csv(f"{OUT}/X_test.csv", index=False)
y_train.to_csv(f"{OUT}/y_train.csv", index=False);  y_test.to_csv(f"{OUT}/y_test.csv", index=False)

np.savez(f"{OUT}/titanic_split.npz",
         X_train=X_train.to_numpy(dtype=float), X_test=X_test.to_numpy(dtype=float),
         y_train=y_train.to_numpy(dtype=int),   y_test=y_test.to_numpy(dtype=int),
         feature_names=np.array(list(X_train.columns), dtype=str))
print("Saved:", sorted(os.listdir(OUT)))

Saved: ['X_test.csv', 'X_train.csv', 'titanic_split.npz', 'y_test.csv', 'y_train.csv']


## Summary
* **Target:** `Survived` (binary) — *predict whether a passenger survived from class, sex, age, family size, fare and embarkation port.*
* **Dropped:** `PassengerId`, `Name`, `Ticket`, `Cabin`.
* **Split:** stratified 80/20, `random_state=42` → 712 train / 179 test rows.
* **Imputation / scaling:** learned on train only (no leakage).
* **Final design matrix:** 9 numeric columns — `Age, SibSp, Parch, Fare, Sex_male, Pclass_2, Pclass_3, Embarked_Q, Embarked_S`.
* **Ready for modelling:** `X_train, X_test, y_train, y_test`.